In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = pd.Series(data.target, name="target")

df = X.copy()
df["target"] = y

df["diagnosis"] = df["target"].map({
    0: "Malignant",
    1: "Benign"
})

print(df.head())
print("\nTarget names:", data.target_names)

In [ ]:
print("Shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
print(df["diagnosis"].value_counts())

print("\nPercentages:")
print(df["diagnosis"].value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(data=df, x="diagnosis")

plt.title("Distribution of Breast Cancer Classes")
plt.xlabel("Diagnosis")
plt.ylabel("Number of Samples")

plt.show()

In [ ]:
selected_features = [
    "mean radius",
    "mean texture",
    "mean perimeter"
]

print(df.groupby("diagnosis")[selected_features].mean())

In [ ]:
for feature in selected_features:
    plt.figure(figsize=(7, 4))

    sns.boxplot(
        data=df,
        x="diagnosis",
        y=feature
    )

    plt.title(f"{feature} by Diagnosis")
    plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

y_prob = model.predict_proba(X_test_scaled)

In [ ]:
print("First 10 predicted classes:")
print(y_pred[:10])

print("\nFirst 10 probability predictions:")
print(y_prob[:10])

In [ ]:
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Probability_Malignant": y_prob[:, 0],
    "Probability_Benign": y_prob[:, 1]
})

results["Actual_Diagnosis"] = results["Actual"].map({
    0: "Malignant",
    1: "Benign"
})

results["Predicted_Diagnosis"] = results["Predicted"].map({
    0: "Malignant",
    1: "Benign"
})

print(results.head(10))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)

# Treat malignant (class 0) as the positive class.
precision = precision_score(
    y_test, y_pred, pos_label=0
)

recall = recall_score(
    y_test, y_pred, pos_label=0
)

f1 = f1_score(
    y_test, y_pred, pos_label=0
)

print("Accuracy:", accuracy)
print("Precision for malignant:", precision)
print("Recall for malignant:", recall)
print("F1-score for malignant:", f1)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Malignant", "Benign"]
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1]
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Malignant", "Benign"]
)

display.plot(cmap="Blues")

plt.title("Confusion Matrix")
plt.show()

In [ ]:
balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

balanced_model.fit(X_train_scaled, y_train)

y_pred_balanced = balanced_model.predict(X_test_scaled)

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision (Malignant)",
        "Recall (Malignant)",
        "F1-score (Malignant)"
    ],
    "Original Model": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, pos_label=0),
        recall_score(y_test, y_pred, pos_label=0),
        f1_score(y_test, y_pred, pos_label=0)
    ],
    "Balanced Model": [
        accuracy_score(y_test, y_pred_balanced),
        precision_score(y_test, y_pred_balanced, pos_label=0),
        recall_score(y_test, y_pred_balanced, pos_label=0),
        f1_score(y_test, y_pred_balanced, pos_label=0)
    ]
})

print(comparison.round(3))

In [ ]:
cm_balanced = confusion_matrix(
    y_test,
    y_pred_balanced,
    labels=[0, 1]
)

display_balanced = ConfusionMatrixDisplay(
    confusion_matrix=cm_balanced,
    display_labels=["Malignant", "Benign"]
)

display_balanced.plot(cmap="Blues")

plt.title("Confusion Matrix — Balanced Model")
plt.show()